In [10]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Load dataset
df = pd.read_csv('IMDB Dataset.csv')

# Initialize tools
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Slang dictionary
slang_dict = {
    "vibe": "good",
    "fire": "amazing",
    "lit": "exciting",
    "goated": "great",
    "meh": "boring",
    "sucks": "bad",
    "lame": "bad",
    "trash": "bad",
    "banger": "amazing",
    "dope": "cool"
}

# Function to replace slang
def replace_slang(text):
    words = text.lower().split()
    return ' '.join([slang_dict.get(w, w) for w in words])

# Text preprocessing
def preprocess_text(text):
    text = replace_slang(text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    text = text.lower()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

# Apply preprocessing
df['clean_review'] = df['review'].apply(preprocess_text)

# Feature extraction
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['clean_review']).toarray()
y = df['sentiment'].map({'positive': 1, 'negative': 0})

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("Model Performance:\n")
print(classification_report(y_test, y_pred))

# Prediction function
def predict_sentiment(review):
    review = replace_slang(review)
    cleaned = preprocess_text(review)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)
    return "Positive" if pred[0] == 1 else "Negative"

# Test case
test_review = "this movie is a vibe"
print(f"\nSample Prediction: \"{test_review}\" → {predict_sentiment(test_review)}")
test_review = "this movie is very fabulous"
print(f"\nSample Prediction: \"{test_review}\" → {predict_sentiment(test_review)}")


Model Performance:

              precision    recall  f1-score   support

           0       0.90      0.87      0.89      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000


Sample Prediction: "this movie is a vibe" → Positive

Sample Prediction: "this movie is very fabulous" → Positive
